# 5A. Optimización de Hiperparámetros (Modelos Base)

## 1. Introducción Teórica
En el cuaderno anterior (`04A`) extrajimos dos conclusiones vitales para la arquitectura de nuestro proyecto:
1. **Para la Premier League**, el modelo debe entrenarse pura y exclusivamente con datos del fútbol inglés.
2. **Para LaLiga**, el modelo mejora su capacidad de generalización cuando se entrena con **Todos los datos (Ambos)**, pero, obviamente, su validez debe evaluarse midiendo su acierto **exclusivamente sobre partidos españoles**.

En este cuaderno dejaremos atrás la exploración y nos centraremos en **exprimir matemáticamente el rendimiento** de nuestros modelos base (Regresión Logística y Random Forest). Para ello, utilizaremos la técnica `GridSearchCV`, que buscará el equilibrio perfecto entre Sesgo y Varianza ajustando los engranajes internos de los algoritmos.

> **Regla de Oro (Reproducibilidad y Fugas):** Inyectaremos `random_state=42` a todas las funciones matemáticas para asegurar resultados reproducibles. Además, obligaremos al *GridSearchCV* a buscar los hiperparámetros usando nuestro `walk_forward_split`, garantizando que el modelo jamás vea el futuro durante su optimización.

In [10]:
import pandas as pd
import numpy as np
import sys
import os
import joblib  # Para exportar los modelos finales

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Crear directorio para modelos si no existe
os.makedirs('../models', exist_ok=True)

# Importar funciones de utilidad propias
sys.path.append(os.path.abspath('../src'))
from model_utils import walk_forward_split, calculate_metrics

import warnings
warnings.filterwarnings('ignore')

## 2. Blindaje Analítico (Carga de Datos)
Mantenemos la censura sobre las variables de futuro para evitar hacer trampas predictivas.

In [11]:
df = pd.read_csv('../data/processed/train_features.csv')

target_col = 'resultado_cod'
col_temporada = 'anio_fin_temporada'

cols_no_predictivas = [
    'id', 'fecha', 'local', 'visitante', 'division', col_temporada,
    'resultado', 'resultado_cod', 'goles_local', 'goles_visitante', 'puntos_local', 'puntos_visitante'
]

## 3. Adaptadores Personalizados para GridSearchCV
Para que el optimizador automático haga exactamente lo que queremos sin cometer errores conceptuales, debemos crear dos "adaptadores":

1. **Generador Cruzado:** Entrena con "Ambos" pero evalúa solo en España. El `GridSearchCV` evaluará ciegamente lo que le pases en el conjunto de test. Este adaptador filtra y borra todos los índices de partidos ingleses en la fase de test.
2. **Scorer de AUC-PR Multiclase:** *Scikit-Learn* no tiene un evaluador nativo para AUC-PR en 3 clases (1, X, 2). Envolveremos nuestra propia función `calculate_metrics` (de nuestro archivo `model_utils.py`) usando `make_scorer` para obligar al Grid a buscar hiperparámetros que suban la métrica oficial del torneo.

In [12]:
def walk_forward_split_cruzado(df, col_temporada, window_type, window_size):
    """
    Generador que retorna Train (Todo) y Test (SOLO LaLiga).
    """
    base_splitter = walk_forward_split(df, col_temporada=col_temporada, window_type=window_type, window_size=window_size)
    
    for train_idx, test_idx in base_splitter:
        test_idx_laliga = df.loc[test_idx][df.loc[test_idx]['division'] == 'ESP_LaLiga'].index
        yield train_idx, test_idx_laliga

def auc_pr_scorer_wrapper(y_true, y_proba, **kwargs):
    """
    Envuelve nuestra función nativa de utilidades. Como GridSearchCV con needs_proba=True 
    solo pasa y_proba, inyectamos y_true como y_pred temporal (solo nos interesa el AUC-PR que devuelve).
    """
    _, auc_pr = calculate_metrics(y_true, y_pred=y_true, y_proba=y_proba)
    return auc_pr

# Creamos el scorer inyectable para el GridSearchCV
auc_pr_scorer = make_scorer(auc_pr_scorer_wrapper, response_method='predict_proba')


## 4. Optimización Pura: PREMIER LEAGUE

Para Inglaterra usamos datos puros ingleses. A través de la etiqueta `'selector'`, daremos **libertad absoluta** al GridSearchCV para que pruebe si es mejor usar `PCA` o dejar pasar los datos en crudo (`passthrough`). Así nunca embotellaremos el rendimiento de forma artificial.

In [13]:
print("===============================================")
print(" OPTIMIZACIÓN PREMIER LEAGUE (Datos Puros) ")
print("===============================================")

df_premier = df[df['division'] == 'ENG_Premier'].copy().reset_index(drop=True)
X_prem = df_premier.drop(columns=cols_no_predictivas)
y_prem = df_premier[target_col]

# Asumimos que la ventana ganadora fue 'expanding' según el cuaderno 04A
cv_prem = list(walk_forward_split(df_premier, col_temporada=col_temporada, window_type='expanding', window_size=1))

# ---------------------------------------------------------
# 4.1 Búsqueda para Logistic Regression
# ---------------------------------------------------------
pipeline_lr_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', 'passthrough'), # Se sustituirá dinámicamente por PCA o nada
    ('model', LogisticRegression(class_weight='balanced', random_state=42))
])

param_grid_lr = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__C': [0.01, 0.1, 1.0, 10.0],
}

grid_lr_prem = GridSearchCV(pipeline_lr_prem, param_grid_lr, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_lr_prem.fit(X_prem, y_prem)

print(f"[LogReg Premier] Mejor AUC-PR Cross-Val: {grid_lr_prem.best_score_:.4f}")
print(f"[LogReg Premier] Mejores Parámetros: {grid_lr_prem.best_params_}\n")

# ---------------------------------------------------------
# 4.2 Búsqueda para Random Forest
# ---------------------------------------------------------
pipeline_rf_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', 'passthrough'),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid_rf = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 7],
}

grid_rf_prem = GridSearchCV(pipeline_rf_prem, param_grid_rf, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_rf_prem.fit(X_prem, y_prem)

print(f"[RF Premier] Mejor AUC-PR Cross-Val: {grid_rf_prem.best_score_:.4f}")
print(f"[RF Premier] Mejores Parámetros: {grid_rf_prem.best_params_}\n")

# EXPORTAMOS EL GANADOR ABSOLUTO DE LA PREMIER
#mejor_modelo_premier = grid_lr_prem.best_estimator_ if grid_lr_prem.best_score_ > grid_rf_prem.best_score_ else grid_rf_prem.best_estimator_
#joblib.dump(mejor_modelo_premier, '../models/best_baseline_premier.pkl')
#print("-> Modelo de la Premier League exportado con éxito a la carpeta 'models/'.")

 OPTIMIZACIÓN PREMIER LEAGUE (Datos Puros) 
[LogReg Premier] Mejor AUC-PR Cross-Val: 0.4853
[LogReg Premier] Mejores Parámetros: {'model__C': 0.01, 'selector': 'passthrough'}

[RF Premier] Mejor AUC-PR Cross-Val: 0.4795
[RF Premier] Mejores Parámetros: {'model__max_depth': 5, 'model__n_estimators': 200, 'selector': 'passthrough'}



## 5. Optimización Híbrida: LALIGA (Train Ambos -> Test LaLiga)

Aquí inyectamos toda la base de datos (Ambos), pero utilizamos nuestra función adaptadora `cv_laliga` para que el `GridSearchCV` mida su exactitud única y exclusivamente cuando predecía el fútbol español.

In [14]:
print("\n===============================================")
print(" OPTIMIZACIÓN LALIGA (Entrenado en Ambos) ")
print("===============================================")

df_ambos = df.copy().reset_index(drop=True)
X_ambos = df_ambos.drop(columns=cols_no_predictivas)
y_ambos = df_ambos[target_col]

cv_laliga = list(walk_forward_split_cruzado(df_ambos, col_temporada=col_temporada, window_type='expanding', window_size=1))

# ---------------------------------------------------------
# 5.1 Búsqueda para Logistic Regression
# ---------------------------------------------------------
param_grid_lr_laliga = {
    'selector': ['passthrough'],
    'model__C': [0.01, 0.1, 1.0, 10.0],
}

param_grid_rf_laliga = {
    'selector': ['passthrough'],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 7],
}

grid_lr_laliga = GridSearchCV(pipeline_lr_prem, param_grid_lr_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_lr_laliga.fit(X_ambos, y_ambos)

print(f"[LogReg LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): {grid_lr_laliga.best_score_:.4f}")
print(f"[LogReg LaLiga] Mejores Parámetros: {grid_lr_laliga.best_params_}\n")

# ---------------------------------------------------------
# 5.2 Búsqueda para Random Forest
# ---------------------------------------------------------
grid_rf_laliga = GridSearchCV(pipeline_rf_prem, param_grid_rf_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_rf_laliga.fit(X_ambos, y_ambos)

print(f"[RF LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): {grid_rf_laliga.best_score_:.4f}")
print(f"[RF LaLiga] Mejores Parámetros: {grid_rf_laliga.best_params_}\n")

# EXPORTAMOS EL GANADOR ABSOLUTO DE LALIGA
# = grid_lr_laliga.best_estimator_ if grid_lr_laliga.best_score_ > grid_rf_laliga.best_score_ else grid_rf_laliga.best_estimator_
#joblib.dump(mejor_modelo_laliga, '../models/best_baseline_laliga.pkl')
#print("-> Modelo de LaLiga exportado con éxito a la carpeta 'models/'.")


 OPTIMIZACIÓN LALIGA (Entrenado en Ambos) 
[LogReg LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4723
[LogReg LaLiga] Mejores Parámetros: {'model__C': 0.01, 'selector': 'passthrough'}

[RF LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4689
[RF LaLiga] Mejores Parámetros: {'model__max_depth': 5, 'model__n_estimators': 200, 'selector': 'passthrough'}



## 6. Conclusión
La optimización fina ha concluido. Has obtenido dos archivos `.pkl` en la carpeta `models/`. 
Estos archivos son tus modelos definitivos. Contienen todo el aprendizaje del pasado encapsulado y configurado matemáticamente a la perfección.

Ya puedes indicarle a tu compañero que comience su cuaderno `05B` para realizar este mismo proceso con sus Redes Neuronales y XGBoost. Cuando él termine y tenga sus propios `.pkl` guardados, podréis reuniros en el cuaderno final para cargar ambos modelos, compararlos con el Test ciego y ver si fusionarlos (Ensemble) os da la victoria.